## Data Insights


This notebook loads `data/processed/publication_data.csv`, produced by `00_data_processing.ipynb`.

In [7]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from IPython.display import HTML

HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Libre+Baskerville:wght@400;700&family=Libre+Franklin:wght@400;500;600;700&display=swap');
</style>
""")

In [8]:
merged_data = pd.read_csv("../data/processed/publication_data.csv")

wdi_cols = [c for c in merged_data.columns if c not in
            ['iso_alpha', 'Year', 'Publications', 'Country', 'data_source_flag']]
years_with_wdi = merged_data.loc[merged_data[wdi_cols].notna().any(axis=1), 'Year']

print(f"Country-year rows: {len(merged_data)}")
print(f"Distinct countries: {merged_data['Country'].nunique()}")
print(f"Years: {merged_data['Year'].min()}-{merged_data['Year'].max()}")
print(f"Years with WDI coverage: {years_with_wdi.min()}-{years_with_wdi.max()} ({years_with_wdi.nunique()} years)")

Country-year rows: 4020
Distinct countries: 201
Years: 2003-2022
Years with WDI coverage: 2003-2022 (20 years)


Window: 2016 - 2022. OSF published reproducibility crisis in 2015, and covid began 2019. This window attempts to capture psych post-reproducibility-crisis-awareness and the landscape pre-covid.

In [9]:
time_interval = merged_data[merged_data['Year'].between(2016, 2022)]

mean_cols = [
    'GDP_per_capita', 'GNI_per_capita_PPP', 'Population', 'Tertiary_enrollment_pct',
    'RnD_expenditure_pct', 'Researchers_per_million',
    'Education_expenditure_pct', 'Tertiary_education_expenditure_pct',
    'Urban_population_pct', 'Sci_tech_articles', 'Internet_users_pct',
]

country_summary = time_interval.groupby(['iso_alpha', 'Country'], as_index=False).agg(
    Publications=('Publications', 'sum'),
    SE_articles_total_3yr=('SE_articles_total', lambda x: x.sum(min_count=1)),
    data_source_flag=('data_source_flag', 'first'),
    **{col: (col, 'mean') for col in mean_cols},
)

country_summary['psych_share'] = country_summary['Publications'] / country_summary['SE_articles_total_3yr']
country_summary['log_publications'] = np.log1p(country_summary['Publications'])

print(f"Countries in 2016-2022 summary: {len(country_summary)}")
print(f"Total psychology publications (2016-2022): {country_summary['Publications'].sum():,.0f}")

Countries in 2016-2022 summary: 201
Total psychology publications (2016-2022): 388,583
